## Experiment Runner

Run the code cells below to run all the experiments performed for this final project. These experiements use pretrained
models and tokenizers with a dataset to classify each observation as privileged or not privileged and provides an
explanation for each prediction.

Use Google Colab GPUs to leverage the compute resources and run the experiments efficiently.

Before running the notebook, clone the project into the Colab runtime so the code, dataset, and outputs all stay under
`/content/NLP_Final_Project`.

In [1]:
import os

PROJECT_DIR = '/content/NLP_Final_Project'

if not os.path.exists(PROJECT_DIR):
    print(f'Project folder not found: {PROJECT_DIR}. Cloning the repo into the Colab runtime.')
    !git clone https://github.com/breburd/NLP_Final_Project

os.chdir(PROJECT_DIR)
print('Current working directory:', os.getcwd())
print('Top-level files:', os.listdir())

Current working directory: /content/NLP_Final_Project
Top-level files: ['tests', '.git', 'models', 'README.md', 'pytest.ini', 'requirements.txt', 'runner.ipynb', '.gitattributes', 'pretrained_models', 'test_emails.json', 'split_dataset.py', 'rename_files.py', 'run_baselines.py', 'preprocess', '.gitignore', 'runner_10_experiments.ipynb', 'ModelDiagram.drawio.png']


In [2]:
import torch
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
print(f"torch.version.cuda: {torch.version.cuda}")

torch.cuda.is_available(): True
torch.version.cuda: 12.1


In [3]:
%matplotlib inline

In [4]:
%pip install -r requirements.txt

In [5]:
# Create the deterministic train/valid/test split once and reuse it across experiments
!python split_dataset.py --data_path preprocess/enron_emails_labeled.csv --output_dir preprocess/splits

TRAIN_PATH = 'preprocess/splits/train.csv'
VALID_PATH = 'preprocess/splits/valid.csv'
TEST_PATH = 'preprocess/splits/test.csv'


Saved deterministic splits to: preprocess/splits
train: 414670 rows (d192b97617a1fd3a6e9f2ad040cc9524bf14638476b59b41a12600d47d633862)
valid: 34493 rows (5afbdbf330252eb93bc1e47490b8351103306724ca97d71c308051f3c4e480db)
test: 68238 rows (98892e7734e0b240c370be5f621b6d0b5d097868311b57ceacec681c4a2a5990)


## Second Experiment Set: 10 BERT Variations

This notebook adds a broader set of experiments for the final project.

The goal is to compare:
- dataset size
- learning rate
- sequence length
- batch size
- number of epochs
- random seed
- pretrained model family

Each experiment writes results to its own folder under `outputs/` so the metrics can be compared later.


In [6]:
from pathlib import Path
import shutil
from google.colab import files

def save_files(directory_name):
  directory = Path(directory_name)
  zip_name = f"{directory.name}_files"
  model_path = directory / "model"

  # Create zip archive
  archive_path = shutil.make_archive(zip_name, 'zip', model_path)

  # Download
  files.download(archive_path)

In [7]:
# Experiment 1: baseline reference run
output_dir = 'outputs/exp01_bert_reference'

!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 256
save_files(output_dir)

2026-05-03 00:12:20.956047: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:12:21.021837: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 330kB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for

In [8]:
# Experiment 2: shorter sequence length for faster training
output_dir = 'outputs/exp02_bert_maxlen128'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 128
save_files(output_dir)

2026-05-03 00:21:23.032886: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:21:23.096748: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [11]:
# Experiment 3: longer sequence length to keep more email context
output_dir = 'outputs/exp03_bert_maxlen512'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 512
save_files(output_dir)


2026-05-03 00:34:41.992452: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:34:42.057445: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [12]:
# Experiment 4: smaller training subset for faster iteration
output_dir = 'outputs/exp04_bert_train15k'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 15000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 256
save_files(output_dir)


2026-05-03 00:49:20.415934: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:49:20.480350: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [13]:
# Experiment 5: larger training subset to test more supervision
output_dir = 'outputs/exp05_bert_train45k'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 45000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 256
save_files(output_dir)


2026-05-03 00:55:07.875701: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:55:07.939863: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [14]:
# Experiment 6: lower learning rate for more conservative updates
output_dir = 'outputs/exp06_bert_lr1e5'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 1e-5 --max_length 256
save_files(output_dir)


2026-05-03 01:07:18.280003: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 01:07:18.344580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [15]:
# Experiment 7: higher learning rate for more aggressive updates
output_dir = 'outputs/exp07_bert_lr3e5'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 3e-5 --max_length 256
save_files(output_dir)


2026-05-03 01:16:15.610534: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 01:16:15.675761: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [16]:
# Experiment 8: larger batch size to test training stability
output_dir = 'outputs/exp08_bert_batch16'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 16 --learning_rate 2e-5 --max_length 256
save_files(output_dir)


2026-05-03 01:25:12.311798: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 01:25:12.375609: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [17]:
# Experiment 9: two epochs to test whether more training helps or overfits
output_dir = 'outputs/exp09_bert_2epochs'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --epochs 2 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 256
save_files(output_dir)


2026-05-03 01:33:29.111713: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 01:33:29.176409: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Moving model to

In [18]:
# Experiment 10: pretrained model comparison with RoBERTa
output_dir = 'outputs/exp10_roberta_base'
!python models/bert_baseline.py --train_path {TRAIN_PATH} --valid_path {VALID_PATH} --test_path {TEST_PATH} --output_dir {output_dir} --model_name roberta-base --epochs 1 --train_size 30000 --valid_size 5000 --test_size 5000 --batch_size 8 --learning_rate 2e-5 --max_length 256
save_files(output_dir)


2026-05-03 01:49:31.738726: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 01:49:31.802830: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Loading the data...
Loading the tokenizer and pretrained model...
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 169kB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `for